In [ ]:
# --- Environment setup: run this cell first (Colab or local) -------------------------
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/cto-school/agentic-ai-engineering.git"   # the public course repository
REPO_DIR = Path("/content/agentic-ai-engineering")

if IN_COLAB:
    if not REPO_DIR.exists():
        print("Cloning the course repository ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
        print("Installing requirements (this takes a minute) ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "setup/requirements-core.txt")], check=True)
    os.chdir(REPO_DIR / "day_03_memory_and_safety")
    # Colab has no .env file. Paste the key you were issued; it is kept only in this runtime.
    from getpass import getpass
    if not os.getenv("OPENROUTER_API_KEY"):
        key = getpass("OPENROUTER_API_KEY (press Enter to stay in mock mode): ").strip()
        if key:
            os.environ["OPENROUTER_API_KEY"] = key
else:
    # Local machine: the key is read from the .env file at the repository root
    # (Day 1.1 explains how to create it from .env.example).
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))

print("Working directory:", os.getcwd())
print("Mode:", "LIVE (OpenRouter key found)" if os.getenv("OPENROUTER_API_KEY") else "MOCK (no key found: deterministic answers, no credit spent)")


# Day 3.1 — Conversation History

## Before you begin

### Learning outcomes

- Explain why a model call cannot remember anything you did not send it.
- Show the forgetting yourself, then fix it by resending the earlier messages.
- Separate conversation history (application state) from persistent memory.

Architecture reference: [Day 3 diagrams D08](../diagrams/source/day_03.md).

### Expected observation

The same question is answered correctly when the earlier messages are sent, and answered with "I don't know" when they are not. In LIVE mode the wording will differ; the behaviour will not.

## Concept briefing

## Three different places information can live

Context is what the model sees in one call. State is information the application carries
while a run is active. Persistent memory is selected data stored for later interactions.
These layers may contain similar text, but their lifecycles and risks differ.

A conversation does not become permanent because it feels continuous. The application
resends earlier messages on every call. If it stops resending them, the model cannot
answer questions about them - not because it forgot, but because it was never told.


### API key reminder

Day 3 runs completely without an API key. If you *do* want the live comparisons, create the
`.env` file exactly as shown in **Day 1.1 — Your First Model Call** (repository root, next to
`README.md`, containing `OPENROUTER_API_KEY=...`). The setup cell below prints which mode you
are in.

In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/safe_task_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

## Step 1 — A conversation is just a list of messages

Nothing is stored on the provider's side between calls. The application keeps a Python list
and sends it again on every request. Let's build that list for a short conversation.

In [ ]:
from safe_task_agent import Message  # a tiny dataclass: role + content

# The list below is the ENTIRE memory of our chatbot. If it is not in this list,
# the model has no way to know it.
history = [
    Message("system", "You are a concise study assistant."),
    Message("user", "My final year project is called Aurora."),
    Message("assistant", "Understood."),
]

for message in history:
    print(f"{message.role:>9}: {message.content}")
print("\nMessages in history:", len(history))

## Step 2 — A stand-in for the model

To see the forgetting clearly we use a five-line local `respond()` function instead of a real
model. It is not intelligent: it can only look at the messages it was handed. That single
limitation is exactly the one a real model call has, which is what makes it a fair stand-in.

In [ ]:
def respond(messages):
    """Answer using ONLY the messages passed in - just like a real model call."""
    project_name = None
    # Everything except the final message is "the history" we were given.
    for message in messages[:-1]:
        if "is called" in message.content:
            # e.g. "My final year project is called Aurora." -> "Aurora"
            project_name = message.content.split("is called")[-1].strip(" .")

    question = messages[-1].content.lower() if messages else ""
    if "name" in question:
        if project_name:
            return project_name
        return "I don't know - the project name is not in the messages you sent me."
    return "(this demo only answers questions about the project name)"

print("respond() is defined. It reads nothing but its `messages` argument.")

## Step 3 — Ask with the full history

We append the question to the history we have been carrying and send the whole list.

In [ ]:
question = Message("user", "Remind me: what is the project name?")

with_history = history + [question]
print("Messages sent :", len(with_history))
print("Answer        :", respond(with_history))

## Step 4 — Ask again, sending only the question

Same function, same question, but this time we forget to resend the earlier turns. This is the
single most common beginner bug: building a chatbot that sends one message per call.

In [ ]:
without_history = [question]           # the fact about Aurora is simply not here
print("Messages sent :", len(without_history))
print("Answer        :", respond(without_history))

print("\nSame question, different answer - the difference is what we sent, not what the model 'knows'.")

## Step 5 — The same two calls against the real model (optional)

If `LIVE` is `True` the cell below repeats Steps 3 and 4 through OpenRouter. If not, it prints
the mock answers instead, so the notebook always runs. Note the `try/except`: one network error
must never stop the lesson.

In [ ]:
import json, urllib.request

def call_openrouter(messages):
    """Smallest possible OpenRouter chat call. Only used when LIVE is True."""
    payload = {
        "model": os.getenv("OPENROUTER_MODEL", "openai/gpt-oss-120b"),
        "messages": [{"role": m.role, "content": m.content} for m in messages],
        "temperature": 0,
        "max_tokens": 60,
    }
    request = urllib.request.Request(
        "https://openrouter.ai/api/v1/chat/completions",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"},
        method="POST")
    with urllib.request.urlopen(request, timeout=60) as response:
        data = json.loads(response.read().decode())
    return data["choices"][0]["message"]["content"].strip()


def ask(messages):
    """LIVE when a key exists, MOCK otherwise, and MOCK again if the network fails."""
    if not LIVE:
        return "MOCK -> " + respond(messages)
    try:
        return "LIVE -> " + call_openrouter(messages)
    except Exception as exc:                      # timeout, 400, quota, anything
        print("Live call failed, falling back to the mock:", exc)
        return "MOCK -> " + respond(messages)

print("With history   :", ask(with_history))
print("Without history:", ask(without_history))

## Step 6 — History is application state, and it grows

Because we resend everything, every turn makes the next call bigger. Watch the size grow.

In [ ]:
growing = list(history)
for turn in range(1, 5):
    growing.append(Message("user", f"Turn {turn}: here is another synthetic project detail."))
    growing.append(Message("assistant", f"Turn {turn}: noted."))
    characters = sum(len(m.content) for m in growing)
    print(f"after turn {turn}: {len(growing):>2} messages, {characters:>4} characters resent")

print("\nNothing trims this list yet. Day 3.2 gives it a budget.")

### Try it yourself

Predict this before you run the cell: we keep the **system** message and the question, but drop
the one user message that contains the project name. Does the answer come back correct?

In [ ]:
# --- Worked solution ---
# Keep the system prompt (it sets the style) but drop the message carrying the fact.
system_only = [history[0], question]         # index 0 is the system message
print("Messages sent:", [m.role for m in system_only])
print("Answer       :", respond(system_only))

# The answer is "I don't know". A system prompt controls TONE, it does not carry FACTS.
# Only the messages you actually resend can be used.

### Checkpoint

**1. Where does the memory of a conversation actually live?**

<details><summary>Show answer</summary>

In the application's own list of messages. The provider stores nothing between calls, so anything you do not resend is gone. "The model forgot" almost always means "my code did not send it".

</details>

**2. Is conversation history the same thing as persistent memory?**

<details><summary>Show answer</summary>

No. History is state that lives for one run and is resent in full. Persistent memory (Day 3.3) is a small set of selected facts saved in a store, retrieved on demand, and editable or deletable by the user.

</details>

### Recap

- **Limitation we saw:** A call answered "I don't know" purely because we did not resend one message.
- **Layer we added:** An explicit, application-owned message history passed into every call.
- **Evidence it worked:** The same question returned "Aurora" with the history and "I don't know" without it.